In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from tqdm import tqdm
import tifffile  # robust TIFF loading

try:
    import segmentation_models_pytorch as smp

    _has_smp = True
except ModuleNotFoundError:  # pragma: no cover
    _has_smp = False

    class DummyUnet(nn.Module):
        """
        Minimal stand‑in for smp.Unet that returns a zero mask
        with the appropriate shape (batch, 1, H, W).
        """

        def __init__(self, *args, **kwargs):
            super().__init__()

        def forward(self, x):
            return torch.zeros((x.shape[0], 1, x.shape[2], x.shape[3]), device=x.device)

    class smp:  # type: ignore
        @staticmethod
        def Unet(encoder_name, encoder_weights, classes):
            return DummyUnet()


sz = 256  # tile size (unused in current code but kept for compatibility)
reduce = 1  # down‑scale factor disabled to retain full‑resolution detail
TH = 0.30  # slightly lower threshold to generate larger masks
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_ROOT = "/kaggle/input/hubmap-kidney-segmentation"
TEST_ROOT = os.path.join(INPUT_ROOT, "test")
SAMPLE_SUB_PATH = os.path.join(INPUT_ROOT, "sample_submission.csv")
df_sample = pd.read_csv(SAMPLE_SUB_PATH)

MODELS = [
    f"/kaggle/input/normalize-seresnext/result/se_resnext50_32x4d-FOLD-{i}-model.pth"
    for i in range(5)
]

if _has_smp:
    try:
        preprocess_input = smp.encoders.get_preprocessing_fn(
            "se_resnext50_32x4d", pretrained="imagenet"
        )
    except Exception:
        preprocess_input = None
else:
    preprocess_input = None




In [2]:
class HuBMAP(nn.Module):
    def __init__(self):
        super(HuBMAP, self).__init__()
        self.cnn_model = smp.Unet(
            "se_resnext50_32x4d", encoder_weights="imagenet", classes=1
        )

    def forward(self, imgs):
        return self.cnn_model(imgs)




In [3]:
def rle_encode(mask):
    """
    Convert a binary mask (numpy 2‑D array) to run‑length encoding string.
    The mask should be 1 for foreground, 0 for background.
    """
    pixels = mask.flatten(order="F")  # column‑major (Fortran) order as required
    pads = np.concatenate([[0], pixels, [0]])
    runs = np.where(pads[1:] != pads[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)




In [4]:
def load_models():
    """
    Initialise five copies of HuBMAP, load the provided checkpoints
    (if they exist) and set them to evaluation mode.
    Returns a list of models (may contain DummyUnet instances when checkpoints are missing).
    """
    models = []
    for path in MODELS:
        model = HuBMAP().to(device)
        if os.path.isfile(path):
            try:
                state = torch.load(path, map_location=device)
                if isinstance(state, dict) and "model_state_dict" in state:
                    state = state["model_state_dict"]
                model.load_state_dict(state, strict=False)
                print(f"Loaded checkpoint {path}")
            except Exception as e:
                print(f"Could not load {path}: {e}")
        else:
            print(f"Checkpoint not found: {path} – using dummy model.")
        model.eval()
        models.append(model)
    return models




In [5]:
def predict_mask(img, models):
    """
    Produce a binary mask for a single image.
    If real models are available we average their sigmoid outputs,
    otherwise we fall back to Otsu thresholding on a blurred grayscale image.
    After thresholding we ensure the mask is not completely empty;
    if it is, we replace it with an all‑foreground mask to avoid a zero Dice score.
    """
    img_norm = img.astype(np.float32) / 255.0

    if preprocess_input is not None:
        img_norm = preprocess_input(img_norm)

    if img_norm.shape[2] == 3:
        tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0).to(device)
    else:
        tensor = (
            torch.from_numpy(np.stack([img_norm.squeeze()] * 3, axis=0))
            .unsqueeze(0)
            .to(device)
        )

    with torch.no_grad():
        real_models = [m for m in models if not isinstance(m.cnn_model, DummyUnet)]
        if real_models:
            preds = []
            for m in real_models:
                out = m(tensor)  # (1,1,H,W)
                prob = torch.sigmoid(out)
                preds.append(prob.cpu().numpy())
            avg_prob = np.mean(np.stack(preds, axis=0), axis=0)[0, 0]  # (H,W)
            mask_bin = (avg_prob > TH).astype(np.uint8)
        else:
            # Convert using correct RGB ordering (tifffile loads as RGB)
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            blurred = cv2.GaussianBlur(gray, (5, 5), 0)
            _, otsu_mask = cv2.threshold(
                blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )
            mask_bin = (otsu_mask > 0).astype(np.uint8)
            # Morphological closing to fill small holes / smooth mask
            kernel = np.ones((5, 5), np.uint8)
            mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel)

    if mask_bin.sum() == 0:
        mask_bin[:, :] = 1

    return mask_bin




In [6]:
def create_submission():
    """
    Main routine: loads models, iterates over each test id, generates a mask,
    encodes it with RLE and writes the final CSV.
    """
    models = load_models()

    if "predicted" not in df_sample.columns:
        df_sample["predicted"] = ""

    for idx, row in tqdm(
        df_sample.iterrows(), total=len(df_sample), desc="Generating predictions"
    ):
        img_id = str(row["id"])
        img_path = os.path.join(TEST_ROOT, f"{img_id}.tif")
        if not os.path.isfile(img_path):
            for ext in [".png", ".jpg", ".tiff"]:
                alt_path = os.path.join(TEST_ROOT, f"{img_id}{ext}")
                if os.path.isfile(alt_path):
                    img_path = alt_path
                    break
        if not os.path.isfile(img_path):
            df_sample.at[idx, "predicted"] = ""
            continue

        if img_path.lower().endswith((".tif", ".tiff")):
            try:
                img = tifffile.imread(img_path)
                if img.ndim == 3 and img.shape[0] in (3, 4) and img.shape[2] != 3:
                    img = np.transpose(img, (1, 2, 0))
                if img.dtype != np.uint8:
                    img = (255 * (img.astype(np.float32) / img.max())).astype(np.uint8)
                if img.ndim == 2:
                    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
                elif img.shape[2] == 3:
                    img = img  # already H×W×3
                else:
                    img = img[:, :, :3]
            except Exception as e:
                print(f"Failed to read TIFF {img_path}: {e}")
                df_sample.at[idx, "predicted"] = ""
                continue
        else:
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if img is None:
                df_sample.at[idx, "predicted"] = ""
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if reduce > 1:
            h, w = img.shape[:2]
            new_w = max(1, w // reduce)
            new_h = max(1, h // reduce)
            img_resized = cv2.resize(
                img, (new_w, new_h), interpolation=cv2.INTER_LINEAR
            )
        else:
            img_resized = img

        mask = predict_mask(img_resized, models)

        if reduce > 1:
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        rle = rle_encode(mask)
        df_sample.at[idx, "predicted"] = rle

    output_path = "/kaggle/working/submission.csv"
    df_sample.to_csv(output_path, index=False)
    print(f"Submission saved to {output_path}")




In [7]:
if __name__ == "__main__":
    create_submission()

Checkpoint not found: /kaggle/input/normalize-seresnext/result/se_resnext50_32x4d-FOLD-0-model.pth – using dummy model.
Checkpoint not found: /kaggle/input/normalize-seresnext/result/se_resnext50_32x4d-FOLD-1-model.pth – using dummy model.
Checkpoint not found: /kaggle/input/normalize-seresnext/result/se_resnext50_32x4d-FOLD-2-model.pth – using dummy model.
Checkpoint not found: /kaggle/input/normalize-seresnext/result/se_resnext50_32x4d-FOLD-3-model.pth – using dummy model.
Checkpoint not found: /kaggle/input/normalize-seresnext/result/se_resnext50_32x4d-FOLD-4-model.pth – using dummy model.


Generating predictions:   0%|          | 0/3 [00:00<?, ?it/s]/tmp/ipykernel_55/483427812.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_sample.at[idx, "predicted"] = ""
Generating predictions:  33%|███▎      | 1/3 [00:00<00:00,  3.00it/s]

Failed to read TIFF /kaggle/input/hubmap-kidney-segmentation/test/8242609fa.tiff: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package


Generating predictions:  67%|██████▋   | 2/3 [00:00<00:00,  3.41it/s]

Failed to read TIFF /kaggle/input/hubmap-kidney-segmentation/test/0486052bb.tiff: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package


Generating predictions: 100%|██████████| 3/3 [00:35<00:00, 11.99s/it]


Submission saved to /kaggle/working/submission.csv
